In [1]:
import difflib
import json
import re
from dataclasses import dataclass

from transformers import BartForConditionalGeneration, BartTokenizer, pipeline  # type: ignore[import]

from core_shared.config import WhitepaperConfig

# ---------------------------------------------------------------------------
# Models — loaded once at import time
# ---------------------------------------------------------------------------

_sentiment_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    truncation=True,
    max_length=512,
)

_summarizer_tokenizer = BartTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6") # type: ignore[assignment]
_summarizer_model = BartForConditionalGeneration.from_pretrained("sshleifer/distilbart-cnn-12-6")

# ---------------------------------------------------------------------------
# Data structures
# ---------------------------------------------------------------------------

@dataclass
class SectionAnalysis:
    """Analysis results for a single document section.

    Attributes:
        word_count: Number of words in the section body.
        sentiment: Dominant sentiment label (positive / negative / neutral).
        score: Formatted score string for all three sentiment classes.
        resume: One-to-two sentence summary of the section.
    """

    word_count: int
    sentiment: str
    score: str
    resume: str


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------


def _normalize(text: str) -> str:
    """Strip section numbering and punctuation, then lowercase and collapse whitespace."""
    text = re.sub(r"^[\d]+[\.\d]*\s*", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.lower().split())


def _truncate_for_model(text: str, max_words: int = 400) -> str:
    """Truncate text to stay within model token limits.

    Args:
        text: Input text to truncate.
        max_words: Maximum number of words to retain.

    Returns:
        Truncated text, or the original if it is already within the limit.
    """
    words = text.split()
    return " ".join(words[:max_words]) if len(words) > max_words else text

# ---------------------------------------------------------------------------
# Analysis helpers
# ---------------------------------------------------------------------------

def _summarize(text: str) -> str:
    """Return a one-to-two sentence extractive summary of the input text."""
    truncated = _truncate_for_model(text, max_words=600)
    if len(truncated.split()) < 30:
        return truncated

    inputs = _summarizer_tokenizer(
        truncated,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    summary_ids = _summarizer_model.generate( # type: ignore[union-attr]
        inputs["input_ids"],
        min_length=20,
        max_length=60,
        do_sample=False,
    )
    return _summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def _analyze_sentiment(text: str) -> tuple[str, str]:
    """Return the dominant sentiment label and a formatted score string.

    Args:
        text: Section body text to analyse.

    Returns:
        A tuple of (dominant_label, score_string) where score_string has the
        form ``"pos=0.87  neg=0.02  neu=0.11"``.
    """
    truncated = _truncate_for_model(text)
    results = _sentiment_pipeline(truncated)[0]  # type: ignore[index]

    scores = {r["label"]: round(r["score"], 2) for r in results} # type: ignore[assignment]
    dominant = max(scores, key=lambda k: scores[k])
    score_str = (
        f"pos={scores.get('positive', 0.0):.2f}  "
        f"neg={scores.get('negative', 0.0):.2f}  "
        f"neu={scores.get('neutral', 0.0):.2f}"
    )
    return dominant, score_str

# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------

def analyze_sections(uuid: str) -> dict[str, dict]:
    """Compute NLP metrics for each section of a whitepaper.

    For each entry in the table of contents, the function locates the
    corresponding body text in the Markdown file and computes word count,
    sentiment, and a short summary.

    Args:
        uuid: UUID of the document to analyse.

    Returns:
        A dictionary mapping section titles to their analysis results::

            {
                "1. Introduction": {
                    "word_count": 312,
                    "sentiment": "positive",
                    "score": "pos=0.87  neg=0.02  neu=0.11",
                    "resume": "...",
                },
                ...
            }

    Raises:
        FileNotFoundError: If the TOC or Markdown file does not exist for the given UUID.
    """
    toc_path = WhitepaperConfig.TOCS_DIR / f"{uuid}.json"
    md_path = WhitepaperConfig.MD_DIR / f"{uuid}.md"

    if not toc_path.exists():
        raise FileNotFoundError(f"TOC not found for UUID: {uuid}")
    if not md_path.exists():
        raise FileNotFoundError(f"Markdown not found for UUID: {uuid}")

    with open(toc_path, encoding="utf-8") as f:
        toc_data = json.load(f)

    md_text = md_path.read_text(encoding="utf-8")
    lines = md_text.splitlines()

    anchors: list[tuple[int, dict]] = []

    for entry in toc_data:
        title = entry["title"]
        norm_full = " ".join(title.lower().split())
        norm_title = _normalize(title)
        best_idx = None
        best_ratio = 0.0

        for i, line in enumerate(lines):
            norm_line = " ".join(line.lower().split())
            norm_line_clean = _normalize(line)

            if norm_line == norm_full:
                best_idx = i
                break

            if norm_title and norm_title in norm_line_clean:
                ratio = difflib.SequenceMatcher(None, norm_title, norm_line_clean).ratio()
                if ratio > best_ratio and ratio >= 0.75:
                    best_ratio = ratio
                    best_idx = i

        if best_idx is not None:
            anchors.append((best_idx, entry))

    if not anchors:
        return {}

    anchors.sort(key=lambda x: x[0])

    results: dict[str, dict] = {}

    for i, (line_idx, entry) in enumerate(anchors):
        start = line_idx + 1
        end = anchors[i + 1][0] if i + 1 < len(anchors) else len(lines)
        content = "\n".join(lines[start:end]).strip()

        if not content:
            continue

        sentiment, score_str = _analyze_sentiment(content)
        resume = _summarize(content)

        results[entry["title"]] = {
            "word_count": len(content.split()),
            "sentiment": sentiment,
            "score": score_str,
            "resume": resume,
        }

    return results

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [2]:
UUID = "c1bbfd0a-20fb-4795-a81b-de7f12541db8"
analyze_sections(UUID)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'Filecoin: A Cryptocurrency Operated File Storage Network': {'word_count': 4,
  'sentiment': 'neutral',
  'score': 'pos=0.03  neg=0.03  neu=0.94',
  'resume': '1e96a1b27a6cb85df68d728cf3695b0c46dbd44d \n\nJuly 15, 2014'},
 'Abstract': {'word_count': 118,
  'sentiment': 'neutral',
  'score': 'pos=0.06  neg=0.02  neu=0.92',
  'resume': ' Filecoin is a distributed electronic currency similar to Bitcoin . Nodes are incentivized to store as much of the entire network’s data as they can . Files are added to the network by spending currency .'},
 '1 Introduction': {'word_count': 335,
  'sentiment': 'neutral',
  'score': 'pos=0.05  neg=0.06  neu=0.89',
  'resume': ' Many computer systems store and access data via commercial service providers . At present, a handful of large providers serve most of these markets . Distributed services, in which agents have individual incentives to store data and optimize local distribution, could provide vastly better solutions .'},
 '2 Design': {'word_count':

In [1]:
import pymupdf

with pymupdf.open("/media/hassan/Programming/Programming/Rochondra/storage/pdfs/8e313d93-a06e-409c-8668-a31b52a34b0a.pdf") as doc:
    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            for line in block["lines"]:
                full = "".join(s["text"] for s in line["spans"])
                if "2.6" in full or "Block Construction" in full:
                    print(f"Page {page.number + 1} | {full!r}")
                    for s in line["spans"]:
                        print(f"  span: {s['text']!r} | font: {s['font']} | size: {s['size']}")

Page 3 | 't (which includes a Bitcoin-style proof-of-work), described in Section 2.6.'
  span: 't' | font: CMMI7 | size: 6.973800182342529
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: '(which' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'includes' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'a' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'Bitcoin-style' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'proof-of-work),' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'described' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'in' | font: CMR10 | size: 9.962599754333496
  span: ' ' | font: CMR10 | size: 9.962599754333496
  span: 'Section' | font: CMR10 | 